# Data Preprocessing


## Setup


In [30]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH = Path("../data/application_train.csv")
raw = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")

Loaded: ..\data\application_train.csv


## Data Cleaning


In [31]:
def clean_known_data_issues(frame):
    cleaned = frame.copy()

    cleaned["DAYS_EMPLOYED_SENTINEL"] = cleaned["DAYS_EMPLOYED"].eq(365243).astype("int8")
    cleaned["DAYS_EMPLOYED"] = cleaned["DAYS_EMPLOYED"].replace(365243, np.nan)

    if "CODE_GENDER" in cleaned.columns:
        cleaned["CODE_GENDER"] = cleaned["CODE_GENDER"].replace("XNA", np.nan)

    return cleaned


cleaned = clean_known_data_issues(raw)
display(cleaned[["DAYS_EMPLOYED", "DAYS_EMPLOYED_SENTINEL"]].describe().T)

sentinel_audit = pd.Series({
    "DAYS_EMPLOYED equals 365243": raw["DAYS_EMPLOYED"].eq(365243).sum(),
    "CODE_GENDER equals XNA": raw["CODE_GENDER"].eq("XNA").sum(),
    "ORGANIZATION_TYPE equals XNA": raw["ORGANIZATION_TYPE"].eq("XNA").sum(),
    "DAYS_BIRTH non-negative": raw["DAYS_BIRTH"].ge(0).sum(),
    "DAYS_ID_PUBLISH positive": raw["DAYS_ID_PUBLISH"].gt(0).sum(),
}, name="count")
display(sentinel_audit.to_frame())

,count,mean,std,min,25%,50%,75%,max
DAYS_EMPLOYED,252137.0,-2384.169325,2338.360162,-17912.0,-3175.0,-1648.0,-767.0,0.0
DAYS_EMPLOYED_SENTINEL,307511.0,0.180072,0.384248,0.0,0.0,0.0,0.0,1.0


,count
DAYS_EMPLOYED equals 365243,55374
CODE_GENDER equals XNA,4
ORGANIZATION_TYPE equals XNA,55374
DAYS_BIRTH non-negative,0
DAYS_ID_PUBLISH positive,0


## Missingness Policy

The confirmed numeric sentinel 365243 in DAYS_EMPLOYED is replaced with NaN and preserved through a dedicated indicator. The four CODE_GENDER values recorded as XNA are also treated as missing; ORGANIZATION_TYPE=XNA is retained because it is common among non-working applicants and may represent a meaningful application-time category rather than a data error.

Features above 40% missing are retained provisionally because many are property attributes whose absence may itself be predictive. Numeric values are median-imputed, categorical values use the training mode, and every feature above 5% missing receives an indicator so the model can distinguish observed values from imputed ones.

In [32]:
missing_policy = (
    cleaned.drop(columns="TARGET")
    .isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "missing_count", "mean": "missing_rate"})
)
missing_policy["missing_pct"] = missing_policy["missing_rate"] * 100
missing_policy["policy"] = pd.cut(
    missing_policy["missing_rate"],
    bins=[-0.001, 0.05, 0.40, 0.60, 1.00],
    labels=[
        "Impute",
        "Impute + missing indicator",
        "Retain provisionally + indicator",
        "Review for removal + indicator",
    ],
)
missing_policy = missing_policy.sort_values("missing_rate", ascending=False)

display(missing_policy.head(30))
display(missing_policy["policy"].value_counts().to_frame("feature_count"))

,missing_count,missing_rate,missing_pct,policy
COMMONAREA_AVG,214865.0,0.698723,69.872297,Review for removal + indicator
COMMONAREA_MODE,214865.0,0.698723,69.872297,Review for removal + indicator
COMMONAREA_MEDI,214865.0,0.698723,69.872297,Review for removal + indicator
NONLIVINGAPARTMENTS_MEDI,213514.0,0.694330,69.432963,Review for removal + indicator
NONLIVINGAPARTMENTS_MODE,213514.0,0.694330,69.432963,Review for removal + indicator
NONLIVINGAPARTMENTS_AVG,213514.0,0.694330,69.432963,Review for removal + indicator
FONDKAPREMONT_MODE,210295.0,0.683862,68.386172,Review for removal + indicator
LIVINGAPARTMENTS_AVG,210199.0,0.683550,68.354953,Review for removal + indicator
LIVINGAPARTMENTS_MEDI,210199.0,0.683550,68.354953,Review for removal + indicator
LIVINGAPARTMENTS_MODE,210199.0,0.683550,68.354953,Review for removal + indicator


,feature_count
policy,
Impute,64
Retain provisionally + indicator,32
Review for removal + indicator,17
Impute + missing indicator,9


## Missingness Indicators


In [33]:
original_feature_cols = [c for c in cleaned.columns if c not in ["TARGET", "SK_ID_CURR"]]
indicator_source_cols = [
    c for c in original_feature_cols
    if cleaned[c].isna().mean() > 0.05 and c != "DAYS_EMPLOYED_SENTINEL"
]

for col in indicator_source_cols:
    cleaned[f"{col}__MISSING"] = cleaned[col].isna().astype("int8")

missing_indicator_cols = [f"{c}__MISSING" for c in indicator_source_cols]
print(f"Created {len(missing_indicator_cols)} missingness indicators.")
display(cleaned[missing_indicator_cols].mean().sort_values(ascending=False).head(20).to_frame("rate"))

Created 58 missingness indicators.


,rate
COMMONAREA_AVG__MISSING,0.698723
COMMONAREA_MODE__MISSING,0.698723
COMMONAREA_MEDI__MISSING,0.698723
NONLIVINGAPARTMENTS_AVG__MISSING,0.694330
NONLIVINGAPARTMENTS_MODE__MISSING,0.694330
NONLIVINGAPARTMENTS_MEDI__MISSING,0.694330
FONDKAPREMONT_MODE__MISSING,0.683862
LIVINGAPARTMENTS_AVG__MISSING,0.683550
LIVINGAPARTMENTS_MEDI__MISSING,0.683550
LIVINGAPARTMENTS_MODE__MISSING,0.683550


## Missingness Model


In [34]:
indicator_model_cols = missing_indicator_cols + ["DAYS_EMPLOYED_SENTINEL"]
X_indicators = cleaned[indicator_model_cols]
y = cleaned["TARGET"]

X_ind_train, X_ind_test, y_ind_train, y_ind_test = train_test_split(
    X_indicators,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

missingness_model = LogisticRegression(
    max_iter=1_000,
    class_weight="balanced",
    random_state=42,
)
missingness_model.fit(X_ind_train, y_ind_train)
missingness_auc = roc_auc_score(
    y_ind_test,
    missingness_model.predict_proba(X_ind_test)[:, 1],
)

print(f"Missingness AUROC: {missingness_auc:.4f}")

Missingness AUROC: 0.5861


## Train-Test Split


In [35]:
X = cleaned.drop(columns=["TARGET", "SK_ID_CURR"])
y = cleaned["TARGET"].copy()
application_ids = cleaned["SK_ID_CURR"].copy()

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    application_ids,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "default_rate_pct": [y_train.mean() * 100, y_test.mean() * 100],
}, index=["train", "test"])
display(split_summary)

,rows,default_rate_pct
train,246008,8.072908
test,61503,8.072777


In [36]:
correlation_threshold = 0.99
minimum_shared_rows = 1000

correlation_candidate_cols = [
    col
    for col in X_train.select_dtypes(include=np.number).columns
    if not col.endswith("__MISSING")
]

numeric_training = X_train[correlation_candidate_cols]

correlation_matrix = numeric_training.corr(
    min_periods=minimum_shared_rows
)

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape, dtype=bool),
        k=1,
    )
)

high_correlation_pairs = (
    upper_triangle.stack()
    .rename("correlation")
    .reset_index()
    .rename(
        columns={
            "level_0": "feature_1",
            "level_1": "feature_2",
        }
    )
)

high_correlation_pairs["absolute_correlation"] = (
    high_correlation_pairs["correlation"].abs()
)

high_correlation_pairs = (
    high_correlation_pairs[
        high_correlation_pairs["absolute_correlation"]
        >= correlation_threshold
    ]
    .sort_values(
        "absolute_correlation",
        ascending=False,
    )
    .reset_index(drop=True)
)

high_correlation_pairs["shared_rows"] = high_correlation_pairs.apply(
    lambda row: numeric_training[
        [row["feature_1"], row["feature_2"]]
    ].dropna().shape[0],
    axis=1,
)

high_correlation_pairs["feature_1_missing_pct"] = (
    high_correlation_pairs["feature_1"]
    .map(numeric_training.isna().mean())
    .mul(100)
)

high_correlation_pairs["feature_2_missing_pct"] = (
    high_correlation_pairs["feature_2"]
    .map(numeric_training.isna().mean())
    .mul(100)
)

display(high_correlation_pairs)
print(f"Found {len(high_correlation_pairs)} pairs at or above {correlation_threshold:.0%}.")

,feature_1,feature_2,correlation,absolute_correlation,shared_rows,feature_1_missing_pct,feature_2_missing_pct
0,FLAG_EMP_PHONE,DAYS_EMPLOYED_SENTINEL,-0.999848,0.999848,246008,0.000000,0.000000
1,OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.998514,0.998514,245197,0.329664,0.329664
2,YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.998391,0.998391,82465,66.478732,66.478732
3,FLOORSMIN_AVG,FLOORSMIN_MEDI,0.997322,0.997322,79087,67.851858,67.851858
4,FLOORSMAX_AVG,FLOORSMAX_MEDI,0.996983,0.996983,123711,49.712611,49.712611
5,ENTRANCES_AVG,ENTRANCES_MEDI,0.996911,0.996911,122233,50.313404,50.313404
6,ELEVATORS_AVG,ELEVATORS_MEDI,0.996319,0.996319,114991,53.257211,53.257211
7,COMMONAREA_AVG,COMMONAREA_MEDI,0.995660,0.995660,74197,69.839599,69.839599
8,LIVINGAREA_AVG,LIVINGAREA_MEDI,0.995472,0.995472,122546,50.186173,50.186173
9,APARTMENTS_AVG,APARTMENTS_MEDI,0.995430,0.995430,121276,50.702416,50.702416


Found 15 pairs at or above 99%.


In [37]:
working_correlation = correlation_matrix.abs().fillna(0.0).copy()
remaining_features = working_correlation.columns.tolist()
correlation_removal_rows = []

while len(remaining_features) > 1:
    active_correlation = working_correlation.loc[
        remaining_features,
        remaining_features,
    ]

    correlation_values = active_correlation.to_numpy(copy=True)
    upper_triangle_rows, upper_triangle_cols = np.triu_indices(
        len(active_correlation),
        k=1,
    )
    upper_triangle_values = correlation_values[
        upper_triangle_rows,
        upper_triangle_cols,
    ]

    if upper_triangle_values.size == 0:
        break

    maximum_position = int(np.argmax(upper_triangle_values))
    maximum_correlation = upper_triangle_values[maximum_position]

    if (
        not np.isfinite(maximum_correlation)
        or maximum_correlation < correlation_threshold
    ):
        break

    feature_1 = active_correlation.index[
        upper_triangle_rows[maximum_position]
    ]
    feature_2 = active_correlation.columns[
        upper_triangle_cols[maximum_position]
    ]

    feature_1_missingness = numeric_training[feature_1].isna().mean()
    feature_2_missingness = numeric_training[feature_2].isna().mean()

    feature_1_mean_correlation = active_correlation.loc[
        feature_1,
        active_correlation.columns != feature_1,
    ].mean()

    feature_2_mean_correlation = active_correlation.loc[
        feature_2,
        active_correlation.columns != feature_2,
    ].mean()

    feature_statistics = pd.DataFrame(
        {
            "feature": [feature_1, feature_2],
            "missingness": [
                feature_1_missingness,
                feature_2_missingness,
            ],
            "mean_absolute_correlation": [
                feature_1_mean_correlation,
                feature_2_mean_correlation,
            ],
        }
    ).sort_values(
        by=[
            "missingness",
            "mean_absolute_correlation",
            "feature",
        ],
        ascending=[False, False, False],
    )

    removed_feature = feature_statistics.iloc[0]["feature"]
    retained_feature = (
        feature_2 if removed_feature == feature_1 else feature_1
    )

    correlation_removal_rows.append(
        {
            "removed_feature": removed_feature,
            "retained_feature": retained_feature,
            "absolute_correlation": maximum_correlation,
            "removed_missingness": numeric_training[
                removed_feature
            ].isna().mean(),
            "retained_missingness": numeric_training[
                retained_feature
            ].isna().mean(),
        }
    )

    remaining_features.remove(removed_feature)

correlation_removal_log = pd.DataFrame(correlation_removal_rows)
correlation_drop_features = correlation_removal_log[
    "removed_feature"
].tolist()

cleaned = cleaned.drop(
    columns=correlation_drop_features,
    errors="ignore",
)
X = X.drop(
    columns=correlation_drop_features,
    errors="ignore",
)
X_train = X_train.drop(
    columns=correlation_drop_features,
    errors="ignore",
)
X_test = X_test.drop(
    columns=correlation_drop_features,
    errors="ignore",
)

missing_indicator_cols = [
    col
    for col in missing_indicator_cols
    if col not in correlation_drop_features
]

indicator_source_cols = [
    col
    for col in indicator_source_cols
    if col not in correlation_drop_features
]

display(correlation_removal_log)
print(f"Dropped {len(correlation_drop_features)} correlated features.")
print(correlation_drop_features)

,removed_feature,retained_feature,absolute_correlation,removed_missingness,retained_missingness
0,FLAG_EMP_PHONE,DAYS_EMPLOYED_SENTINEL,0.999848,0.000000,0.000000
1,OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.998514,0.003297,0.003297
2,YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.998391,0.664787,0.664787
3,FLOORSMIN_AVG,FLOORSMIN_MEDI,0.997322,0.678519,0.678519
4,FLOORSMAX_AVG,FLOORSMAX_MEDI,0.996983,0.497126,0.497126
5,ENTRANCES_AVG,ENTRANCES_MEDI,0.996911,0.503134,0.503134
6,ELEVATORS_AVG,ELEVATORS_MEDI,0.996319,0.532572,0.532572
7,COMMONAREA_MEDI,COMMONAREA_AVG,0.995660,0.698396,0.698396
8,LIVINGAREA_AVG,LIVINGAREA_MEDI,0.995472,0.501862,0.501862
9,APARTMENTS_AVG,APARTMENTS_MEDI,0.995430,0.507024,0.507024


Dropped 15 correlated features.
['FLAG_EMP_PHONE', 'OBS_30_CNT_SOCIAL_CIRCLE', 'YEARS_BUILD_AVG', 'FLOORSMIN_AVG', 'FLOORSMAX_AVG', 'ENTRANCES_AVG', 'ELEVATORS_AVG', 'COMMONAREA_MEDI', 'LIVINGAREA_AVG', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'LIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_AVG', 'LANDAREA_MEDI']


## Preprocessing Pipeline

The preprocessing transformer is fitted only on the training split to prevent validation information from influencing imputers, scaling parameters, or category vocabularies. Median and mode imputation are robust, reproducible defaults for this heterogeneous application dataset, while one-hot encoding avoids imposing an artificial order on nominal categories.

In [38]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.30,
    verbose_feature_names_out=False,
)

print(f"Numeric inputs: {len(numeric_features)}")
print(f"Categorical inputs: {len(categorical_features)}")

Numeric inputs: 148
Categorical inputs: 16


In [39]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
processed_feature_names = preprocessor.get_feature_names_out()

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")
print(f"Output feature count: {len(processed_feature_names):,}")

Processed training shape: (246008, 287)
Processed test shape: (61503, 287)
Output feature count: 287


## Save Processed Data

The cleaned application table and immutable split assignments are saved here; notebook 03 adds application and bureau features before writing the curriculum-compatible preprocessed_train.csv. Keeping the split keyed by SK_ID_CURR ensures every later notebook evaluates the same untouched holdout population.

In [40]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_assignments = pd.DataFrame({
    "SK_ID_CURR": id_train.to_numpy(),
    "TARGET": y_train.to_numpy(),
    "SPLIT": "train",
})

test_assignments = pd.DataFrame({
    "SK_ID_CURR": id_test.to_numpy(),
    "TARGET": y_test.to_numpy(),
    "SPLIT": "test",
})

split_assignments = pd.concat(
    [train_assignments, test_assignments],
    ignore_index=True,
)

split_path = OUTPUT_DIR / "split_assignments.parquet"
split_assignments.to_parquet(split_path, index=False)

print(f"Saved split assignments: {split_path}")

clean_path = OUTPUT_DIR / "application_train_clean.parquet"
cleaned.to_parquet(clean_path, index=False)

manifest = {
    "source": "Home Credit Default Risk/application_train.csv",
    "rows": int(len(cleaned)),
    "columns_after_cleaning": int(cleaned.shape[1]),
    "target": "TARGET",
    "identifier": "SK_ID_CURR",
    "sentinel_replacements": {"DAYS_EMPLOYED": 365243},
    "missing_indicator_sources": indicator_source_cols,
    "numeric_input_count": len(numeric_features),
    "categorical_input_count": len(categorical_features),
    "split_random_state": 42,
    "test_size": 0.20,
}

manifest_path = OUTPUT_DIR / "preprocessing_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

print(f"Saved cleaned data: {clean_path}")
print(f"Saved manifest: {manifest_path}")

Saved split assignments: ..\data\split_assignments.parquet
Saved cleaned data: ..\data\application_train_clean.parquet
Saved manifest: ..\data\preprocessing_manifest.json
